# Multimodal RAG with LangChain

This notebook shows multimodal RAG on table, chart, and text extraction from a sample PDF using **LangChain** and the **NeMo Retriever Library**.

**Note:** This notebook uses the **NeMo Retriever Library** (`nemo-retriever`) with local Hugging Face inference and **LanceDB** (default `./lancedb`, table `nv-ingest`). Install from the [NeMo-Retriever](https://github.com/NVIDIA/NeMo-Retriever) repository (`pip install "nemo-retriever[local]"` plus the CUDA 13 torch stack in [nemo_retriever/README.md](https://github.com/NVIDIA/NeMo-Retriever/blob/main/nemo_retriever/README.md)). Legacy NV-Ingest microservices, Docker Compose, `nv_ingest_client`, and Milvus are not part of the 26.05 customer path.

%pip install -qU langchain langchain_community langchain-nvidia-ai-endpoints "nemo-retriever[local]"

In [ ]:
Extract tables and charts from a sample PDF, embed them, and upload to LanceDB with `GraphIngestor`.

from pathlib import Path

from nemo_retriever.graph_ingestor import GraphIngestor
from nemo_retriever.params import EmbedParams, ExtractParams, VdbUploadParams

pdf = Path("../data/multimodal_test.pdf")
embed_params = EmbedParams(
    model_name="nvidia/llama-nemotron-embed-1b-v2",
    local_ingest_embed_backend="hf",
)

ingestor = (
    GraphIngestor(run_mode="inprocess", documents=[str(pdf.resolve())])
    .extract(
        ExtractParams(
            extract_text=False,
            extract_tables=True,
            extract_charts=True,
            extract_images=False,
        )
    )
    .embed(embed_params)
    .vdb_upload(
        VdbUploadParams(
            vdb_op="lancedb",
            uri="./lancedb",
            table_name="nv-ingest",
            overwrite=True,
        )
    )
)
ingestor.ingest()

In [ ]:
Query LanceDB through `nemo_retriever.retriever.Retriever`, then pass retrieved chunks into a LangChain prompt.

from nemo_retriever.retriever import Retriever

retriever = Retriever(
    lancedb_uri="lancedb",
    lancedb_table="nv-ingest",
    embedder="nvidia/llama-nemotron-embed-1b-v2",
    top_k=5,
    local_query_embed_backend="hf",
)


def retrieve_context(question: str) -> str:
    hits = retriever.query(question)
    return "\n\n".join(h.get("text", "") for h in hits if h.get("text"))

In [13]:
# Retrieval uses `Retriever` from the previous cell (LanceDB). No Milvus vector store is required.

Then, we'll create an RAG chain using [llama-3.1-405b-instruct](https://build.nvidia.com/meta/llama-3_1-405b-instruct) that we can use to query our pdf in natural language

In [3]:
import os 
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# TODO: Add your NVIDIA API key
os.environ["NVIDIA_API_KEY"] = "[YOUR NVIDIA API KEY HERE]"

llm = ChatNVIDIA(model="meta/llama-3.1-405b-instruct")

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Keep the answer concise."
    "\n\n"
    "{context}"
    "Question: {question}"
)

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": lambda q: retrieve_context(q), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

And now we can ask our pdf questions

In [16]:
rag_chain.invoke("What is the dog doing and where?")

'The dog is chasing a squirrel in the front yard.'